# Integrated Intelligent Systems: Tutorial 3 - Body Motion Problem (Scientific Object) (3 Pts)

This notebook introduces a formulation body motion problem, central to designing and implementing Cognition-Enabled Robot Agents, as a description logic system.

## Part 1: Robot-World Manipulation System as Scientific Object (E1–E6) (1.5 Pt)

**Title:** Formalizing the Robot Agent: From Architecture to Description Logic Reasoners

![Alternative Text](RWMS.png)

### Scenario Background
An autonomous robotic agent (the PR2 in the EASE apartment) is deployed to execute a manipulation task ("Serve Eva a drink"). To validate this agent scientifically, we must formalize the system as a **Robot-World Manipulation System (RWMS)**, separate the controlling subsystem from the controlled body-world system, define trajectories as trees, and assess runs against an external problem standard using a description logic reasoner.

---

### Multi-Question Student Task Sheet (0.25 Pt x 6)


1. **E1 · Architectural Cut & Subsystem Separation**
   - In the graphical representation of the RWMS, a strict architectural "cut" is placed between the manipulation executive and the body-world system ($[B \bowtie W]$). Explain the nature of this cut: why is the manipulation executive classified as a `dul:Agent` (non-physical program) while the body-world system is classified under physical objects, and how do they interact exclusively across the boundary?


**Answer E1:**
The cut separates **information** from **physics**. The manipulation executive is a program: it consists of beliefs, plans and decisions, has no mass or location, and is therefore a non-physical `dul:Agent`. The body-world system $[B \bowtie W]$ consists of the robot body and the kitchen. It has mass and geometry, obeys physical laws, and is therefore classified under `dul:PhysicalObject`.
They interact **only across the boundary**: the executive sends **commands** (motor or controller set-points) to the body, and the body sends back **sensor data** (joint states, camera images, forces). The executive cannot move anything directly, and the world can only affect the executive through what is sensed.


2. **E2 · Forward-Branching Time & World Choices**
   - During an operation on a fridge door or container seal, the same action token (e.g., a position-controlled pull) can lead to completely divergent physical outcomes depending on world choices (e.g., seal releases vs. seal holds tight). Draw or describe the structure of the resulting courses, and explain why the possible courses form a tree that branches *only forward* from the initial state $x_0$.


**Answer E2:**
```
                 ┌─ seal releases ──> door opens ──> ...
x0 ── pull ──────┤
                 └─ seal holds ─────> gripper slips / door stays shut ──> re-grasp ──> ...
```
The same action token leads to different successor states depending on the **world's choice**. Each later action branches again, so the possible courses form a tree. It branches **only forward** because the past is fixed: every state has exactly **one history** back to $x_0$ (one parent), while its future is still open. Two branches never merge back into one past.


3. **E3 · Trajectory Classification & Priority Hierarchies**
   - The problem standard $P$ sorts every physical trajectory into one of four classes: **SOL** (Solution), **REPD** (Repaired), **NON** (Nonachievement), and **PROH** (Prohibited). Consider two runs that both successfully place the drink within Eva's reach, but where Run A tears off the container handle (`P4` violation) while Run B maintains licensed contact. Classify both runs and explain why the classifier grades the *entire course* rather than just the final end-state.


**Answer E3:**
- **Run A → PROH:** tearing off the handle violates `P4`, a prohibition, so the run is prohibited even though the drink reached Eva.
- **Run B → SOL:** goal reached with licensed contact throughout (it would be REPD if a failure had occurred and been recovered along the way).

Both runs have the **same end state**, so only the course can tell them apart. Many constraints (no damage, no spill, licensed contact) apply *along* the trajectory. Grading only the end state would wrongly count Run A as a solution.


4. **E4 · Description Logic Partonomy & DUL Constraints**
   - When formalizing the RWMS in description logic under DOLCE Ultra Lite (DUL), we define an RWMS as having *exactly one* executive and *exactly one* body-world system (`RWMS = (=1 hasComponent.ManipExecutive) ⊓ (=1 hasComponent.BodyWorldSystem)`). Using the transitivity of `dul:isPartOf`, explain how a reasoner (such as HermiT) automatically entails that the PR2's head camera is a component of the RWMS without explicitly hardcoding every part-whole link.


**Answer E4:**
Only local part-whole links are asserted:
`headCamera isPartOf pr2Head`, `pr2Head isPartOf pr2`, `pr2 isPartOf bodyWorld`, `bodyWorld isPartOf rwms`
(the last follows from `rwms hasComponent bodyWorld`, since `hasComponent ⊑ hasPart` and `isPartOf` is its inverse).
`dul:isPartOf` is declared **transitive**, so HermiT computes the chain and entails `headCamera isPartOf rwms` itself. No one has to hardcode that link, and any newly added part is covered automatically.


5. **E5 · Ontological Inconsistency & Entailment Limits**
   - An inconsistency in description logic serves as a strict analytical tool. Formulate a competency question that results in an ontological inconsistency when attempting to classify the *manipulation problem itself* as a physical component inside the RWMS. Furthermore, explain why numerical physical laws (such as Coulomb friction $2 \cdot \mu \cdot F \ge m \cdot (g + a)$) must be handled outside description logic by the semantic digital twin's causal model.


**Answer E5:**
**Competency question:** *"Is the manipulation problem `serveDrinkProblem` a component of `rwms`?"*
Axioms: `ManipulationProblem ⊑ dul:Description` (a social object); `RWMS ⊑ ∀hasComponent.(ManipExecutive ⊔ BodyWorldSystem)` with `BodyWorldSystem ⊑ dul:PhysicalObject`; and DUL makes `SocialObject` disjoint from `PhysicalObject`.
Asserting `rwms hasComponent serveDrinkProblem` would force it to be both a Description and a physical/agent component, so the reasoner reports **inconsistency**. This shows the problem standard $P$ is **external** to the system it judges.

**Numerical laws:** DL has no arithmetic over continuous variables (no products like $2\mu F$ and no inequalities between variables). It works with classes and relations, not equations. Checking $2\mu F \ge m(g+a)$ therefore needs numbers and a computation, which the **semantic digital twin's causal/physics model** provides.


6. **E6 · Strength vs. Truth in Agent Validation**
   - The design thesis dictates that an agent decides within its believed causality, which can be tested for logical *strength* via competency queries and for *truth* via physical experiments. Compare how an axiom set that lacks a friction law ($\text{K}_0$) performs on competency queries versus an axiomatization containing explicit dynamical laws ($\text{K}_1/\text{K}_2$), and state what kind of test is required to validate the actual physical truth of those beliefs.


**Answer E6:**
- **K₀ (no friction law):** answers structural queries (what is part of what, which motion causes which state change) but **cannot decide** queries such as *"will this grasp hold the carton?"*. Under the open-world assumption the answer is *unknown*, so K₀ is logically **weaker**.
- **K₁/K₂ (with dynamical laws):** **entail** answers to more queries, e.g. "grasp force F is (in)sufficient for this mass and acceleration". They are logically **stronger**.

Strength is not truth: a stronger axiom set can still be wrong (e.g. a wrong μ for a wet carton). Truth requires **physical experiments**: run the predicted action on the real robot (e.g. lift the carton with force F) and compare the observed outcome (slip / no slip) with the prediction.


## Part 2: Robot Motion Problem and Description Logics (1.5 Pt)

**Title:** Formulating and Querying the Robot Motion Problem with Knowledge Bases & Description Logics

![Alternative Text](PR2.png)

### Scenario Background
Using the interactive Jupyter notebook and RViz simulation environment (`bmp.rviz`, `setup_demo()`), the PR2 mobile manipulator interacts with the EASE apartment model (including the fridge and various kitchen drawers). Instead of hardcoding trajectories, the robotic executive queries a knowledge base (represented via Description Logics / Prolog reasoning) to link task requests to state changes, causes (motions), and capabilities.

[Go to the programming Lab](https://binder.intel4coro.de/v2/gh/maltehue/giskard-examples/bmp_vrb?urlpath=/lab/tree/notebooks/bodyMotionProblem.ipynb)
---

### Multi-Question Student Task Sheet (0.25 Pt x 6)


1. **E7 · Knowledge Base Grounding & Task Requests**
   - Execute the initial query `taskRequest(Request, StateChange).`Examine the resulting solution bindings (e.g., `{'Request': 'open_fridge', 'StateChange': 'open(fridge)'}` and various drawer operations). Explain how mapping abstract task requests to discrete state changes forms the top-level specification layer for the manipulation executive.


**Answer E7:**
The query returns pairs such as `open_fridge → open(fridge)` and `open_drawer_x → open(drawer_x)`. Each request is linked to the **goal state** it is meant to produce. This is the top-level specification layer: it says *what* must be true afterwards, not *how* to achieve it. The executive takes the state change as its goal and chooses the motion later, so the request vocabulary is decoupled from its realization.


2. **E8 · Extending Queries for Causal Motions & Parameters**
   - Extend the query to incorporate motion causes: `taskRequest(Request, StateChange), causes(Motion, StateChange, MotionParam).` Explain how the reasoner bridges the gap between a desired symbolic state change and continuous motion parameters (such as `envJointGoal`, target joint angles, and handle identifiers).


**Answer E8:**
The shared variable `StateChange` joins the two predicates. For each goal state, `causes/3` looks up a motion that produces it, together with its parameters taken from the environment model. For `open(fridge)` these are: target joint value `0.8` (rad), joint `fridge_door_joint`, handle `fridge_door_handle`. The reasoner turns the symbolic goal into a concrete, parameterized motion goal (`envJointGoal`) that a motion controller can execute.


3. **E9 · Integrating Robot Capabilities (`canPerform`)**
   - Finalize the query by adding the robot capability constraint: `taskRequest(Request, StateChange), causes(Motion, StateChange, MotionParam), canPerform(Robot, Motion, MotionParam).`Describe what role the `Robot` variable (`'pr2'`) plays in filtering feasible trajectories through the reasoner.


**Answer E9:**
`canPerform` is a **feasibility filter**. Binding `Robot = 'pr2'` keeps only those motion/parameter combinations the PR2 can actually execute (given its arms, grippers, reach and mobile base). Combinations it can't perform give no solution and are dropped. A different robot would get a different set of feasible motions from the same knowledge base.


4. **E10 · Specializing Request Scope (`open_fridge` vs. `open_anything`)**
   - Modify the query request parameter from general categories like `open_anything` to specifically target `open_fridge`. Explain how refining the scope in the knowledge base reduces the solution search tree and prevents unauthorized or unintended state changes in the kitchen environment.


**Answer E10:**
With `open_anything`, the query unifies with every openable object (fridge and all drawers), so the search tree has many branches. Fixing `Request = open_fridge` leaves only the branch for `open(fridge)`. This makes the search smaller and faster, and it means the only state change the robot may bring about is the requested one. No drawer is opened by accident.


5. **E11 · Visualization and Simulation Linkage in RViz**
   - With the background shell command executing `rviz -d .../bmp.rviz` and displaying the PR2 interacting with the fridge handle, explain how the motion parameters derived from the reasoner's output (`[0.8, b'fridge_door_joint', b'fridge_door_handle']`) are executed as physical joint trajectories in the simulation.


**Answer E11:**
The parameters `[0.8, b'fridge_door_joint', b'fridge_door_handle']` (the `b''` byte strings come from the Prolog-Python interface) are sent as an `envJointGoal` to the motion planner/controller (Giskard). Giskard treats "gripper holds `fridge_door_handle`" and "`fridge_door_joint` reaches 0.8 rad" as constraints, plus collision avoidance. From these it computes a **whole-body joint trajectory** for the PR2 and executes it in simulation. The resulting joint states (robot and door) are published, and RViz (`bmp.rviz`) displays the robot and apartment models moving along them.


6. **E12 · Ontological Limits & Missing Dynamics**
   - While the knowledge base successfully infers *which* motion achieves a state change, explain why a pure Description Logic or symbolic Prolog reasoner cannot verify whether a specific grasp force will cause a wet carton to slip, and identify which subsystem must handle those physical calculations.


**Answer E12:**
Whether a wet carton slips depends on **continuous quantities**: friction μ (lowered by wetness), grasp force F, mass m and acceleration a, through $2\mu F \ge m(g+a)$. DL and Prolog reasoners work with symbols and relations. They have no physical model or measured values to evaluate this inequality, and cannot sense wetness. This must be handled by the **semantic digital twin's causal/physics model** (simulation), together with the **perception and control** subsystem (force/tactile sensing during a test lift).


# Submission guidelines

- Deadline: 24.11.2026, 4 PM
- Submission as Git repo commits pushed within deadlines (If not yet, create a repo with the name **iis-assignment-win26** for the whole semester and invite me as collaborator)
- Put the textual answers in notebook and all related media (images, rules, etc) should then be put into a folder called **Assignment3**. 